# 03 — Review validation and preprocessing

Prepared 9 November 2026 (Asia/Calcutta). **Generated, unexecuted notebook.**

Inspect frozen review-reference contracts and characterize train text without changing the original corpus.

Inputs: existing Phase 1 IMDb manifests and independently verified manifest report; later permitted movie-linked annotations.

IMDb remains binary. A separate adjudicated three-class corpus supplies neutral labels. Train, tuning, calibration and test remain isolated by duplicate/source-movie groups.

No datasets, dependencies or model weights are fetched. No model is trained by running all cells with the default flag. Fill only real, approved local paths. Do not store private review text in committed outputs. Source presence is not Phase 2/3 acceptance.

Read [the Phase 3 runbook](../documentation/phase3/RUNBOOK.md) first.

In [ ]:
from pathlib import Path
import sys

# Explicit opt-in for later local inspection; this delivery has not executed cells.
RUN_APPROVED_EXPERIMENTS = False
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "ARCHITECTURE.md").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the D:/BDA_TA workspace.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workspace:", ROOT)
print("Experiment execution enabled:", RUN_APPROVED_EXPERIMENTS)

## Protocol

1. Review source provenance, duplicate/leakage exclusions and label policy.
2. Divide only the verified validation references into tuning and calibration groups using the shared script.
3. Validate four explicit role manifests through the shared loader.
4. Inspect train-only lengths/scripts/symbol counts; retain immutable original text and aligned feature views.
5. Preserve the official test and every raw file. This notebook does not create labels or movie mappings.

In [ ]:
CONFIG_PATH = ROOT / "configs/training/imdb_binary_tfidf.yaml"
VERIFICATION_PATH = ROOT / "artifacts/data_quality/imdb-manifest-verification-20261109-bdf9f8fa/manifest_verification.json"

if RUN_APPROVED_EXPERIMENTS:
    from ml.common.artifacts import read_json
    from ml.common.config import load_config
    from ml.common.data import load_experiment
    verification = read_json(VERIFICATION_PATH)
    print("Independent verification status:", verification.get("status"))
    config = load_config(CONFIG_PATH)
    bundle = load_experiment(config)
    print(bundle.summary())  # Counts/hashes only; never print raw reviews.
else:
    print("Validation/metadata inspection is disabled; fill tuning/calibration paths first.")

## Prepare validation partitions later

Run from the repository root only after reviewing the Phase 1 verification report:

```powershell
.\.venv\Scripts\python.exe -B scripts/prepare_sentiment_splits.py --validation-manifest 'artifacts/data_quality/imdb-20261108-full-fe07af53/validation.jsonl' --verification-report 'artifacts/data_quality/imdb-manifest-verification-20261109-bdf9f8fa/manifest_verification.json' --output 'data/processed/sentiment/imdb-splits-001'
```

The script writes a fresh output directory; it does not edit train/test or raw reviews. Insert the generated tuning/calibration paths into a reviewed experiment config. No split has been generated by this notebook.

In [ ]:
if RUN_APPROVED_EXPERIMENTS:
    from collections import Counter
    from ml.informal_language.preprocess import observations, feature_view
    histogram = Counter()
    inspected = 0
    for record in bundle.records["train"][:128]:
        text = record.read_text()  # Bounded, hash-checked original.
        details = observations(text)
        histogram["reviews"] += 1
        histogram["code_points_total"] += details["code_points"]
        histogram["symbol_runs_total"] += len(details["symbols"])
        inspected += 1
        view = feature_view(text)
        assert view.original == text and view.text == text
    print({"bounded_train_sample": inspected, "summary": dict(histogram)})
else:
    print("Train-text inspection is disabled; no corpus text was read.")

## Questions for the report

What proportion of sampled training reviews exceeds the selected model token budget? Character counts are not tokenizer counts: calculate actual token lengths only with the approved local tokenizer. Do source-movie and normalized-text groups remain disjoint? How many examples were excluded and why? Describe near-duplicate limitations rather than claiming perfect leakage elimination.

## Interpretation and acceptance

Record measured results, source/model licenses, exact config and manifest hashes, code revision, dependency versions, hardware and unsupported cohorts after the approved runs. Keep the final held-out set separate from model selection. No result has been measured in this notebook. Update the model card from actual reports; missing evidence remains missing.